# Finger IMU practice

2-joint finger model (`assets/finger_2link.xml`): a fixed base segment plus two hinge-jointed phalanges, each carrying an accelerometer + gyro site (`i1`, `i2`). Exploratory MuJoCo work, mirrors `mujoco_practice.ipynb` — not part of the `erp` package.

`assets/finger_2link.xml` is a **`str.format` template, not a loadable XML**. Every number with physical meaning — link lengths, joint damping, servo gains, stall torque, rotor armature, motor time constant, timestep — lives in the Python dicts below; the XML holds only topology and the slots those values go into. Loading it with `mj.MjModel.from_xml_path` will fail, by design: go through `build_xml()`.

In [2]:
import mujoco as mj
from pathlib import Path
import mujoco.viewer
import time
import numpy as np

In [ ]:
XML_RELATIVE_PATH = Path("assets/finger_2link.xml")

def resolve_model_path(relative_path: Path) -> Path:
    """Busca el XML sin depender de desde donde se lanzo Jupyter."""
    candidates = [
        Path.cwd() / relative_path,
        Path.cwd().parent / relative_path,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"No se encontro '{relative_path}' (cwd actual: {Path.cwd()}). "
        "Corre el notebook desde la raiz del proyecto ERP, o ajusta XML_RELATIVE_PATH."
    )

# El archivo es un TEMPLATE: no se puede cargar con mj.MjModel.from_xml_path, hay que
# pasarlo por build_xml() (mas abajo). encoding explicito para no depender del locale.
model_path = resolve_model_path(XML_RELATIVE_PATH)
xml_template = model_path.read_text(encoding="utf-8")
print(f"Cargando template MuJoCo desde: {model_path}")

In [ ]:
# ============================================================================
# PARAMETROS DEL MODELO. Unica fuente de verdad: el XML es solo topologia.
# ============================================================================
# Regla: si un numero tiene significado fisico, va aca y NO en el XML. Asi se
# puede barrer un parametro (kp, tau, largo de falange) sin tocar el template,
# y el mismo template compila las dos variantes que necesita el EKF (ver abajo).

# Geometria del dedo (metros) y damping estructural de los joints.
# i1/i2 = posicion del sitio IMU a lo largo de cada falange (aca, el punto medio).
geometry = dict(
    l0=0.030, w0=0.008,             # base
    l1=0.035, w1=0.006, i1=0.0175,  # falange proximal
    l2=0.030, w2=0.005, i2=0.0150,  # falange distal
    b=0.002,                        # damping viscoso de ambos joints, N*m*s/rad
)

# Opciones del integrador. implicitfast es necesario para que el termino kv del
# servo se integre implicitamente; con el Euler explicito por defecto, ganancias
# de servo realistas vuelven la sim inestable.
sim_options = dict(
    timestep=0.002,            # s
    gravity="0 0 -9.81",       # m/s^2, marco mundo
    integrator="implicitfast",
)

# Recorrido mecanico de cada joint (grados). El ctrlrange del actuador se deriva
# de aca en radianes -> imposible que se desincronicen.
joints = {
    "joint_1": dict(range_deg=(-90.0, 90.0)),
    "joint_2": dict(range_deg=(-120.0, 120.0)),
}

# Servos de posicion. force = kp*(act - angulo) - kv*velocidad.
#   kp, kv     ganancias del PD interno del servo (N*m/rad, N*m*s/rad).
#              kv ~ amortiguamiento critico para I_efectiva = I_eslabon + armature.
#   force_max  torque de stall (N*m) -> forcerange = (-force_max, +force_max).
#   armature   inercia del rotor reflejada por la reductora (kg*m^2). En un servo
#              con reduccion alta esto DOMINA sobre la inercia del eslabon.
#   damping    friccion viscosa de la caja reductora (N*m*s/rad). Se SUMA al
#              damping estructural del joint (geometry["b"]).
#   tau        constante de tiempo del motor (s): ancho de banda finito, el
#              setpoint no salta. Es lo unico que se apaga en el modelo del EKF.
actuators = {
    "act_joint_1": dict(  # proximal: motor mas grande, mueve toda la cadena distal
        joint="joint_1", kp=2.0, kv=0.025,
        force_max=0.30, armature=4e-5, damping=1e-3, tau=0.004,
    ),
    "act_joint_2": dict(  # distal: motor mas chico, carga mucho menor
        joint="joint_2", kp=0.8, kv=0.010,
        force_max=0.15, armature=2e-5, damping=8e-4, tau=0.004,
    ),
}

## Dual model: one template, two compiled models

`assets/finger_2link.xml` is a **template**, not a valid XML — every physically meaningful number is a `{field}` filled from the dicts above. That buys us the *dual model* architecture the EKF needs, from a single source of truth:

| | `model_sim` (plant) | `model_ekf` (filter) |
|---|---|---|
| actuator dynamics | `dyntype="filterexact"`, `tau = 4 ms` | none |
| `na` | 2 | 0 |
| `nx = 2*nv + na` | **6** — $x = [q_1, q_2, v_1, v_2, a_1, a_2]^T$ | **4** — $x = [q_1, q_2, v_1, v_2]^T$ |
| used for | `mj_step`, synthetic IMU data | `mjd_transitionFD`, `mjd_inverseFD` |

**Why two models and not one 6×6 sliced down to 4×4.** In the 6-state plant the control $u$ enters only through the activations $a$; the rigid body sees $a$, not $u$. Slicing $B_{6\times2}$ to its first four rows therefore does *not* give the rigid-body $B$ — it gives that $B$ scaled by the filter's one-step gain $1 - e^{-\Delta t/\tau} \approx 0.39$, and slicing $A$ throws away the $a$ columns that carried the rest of the effect. (With `actearly="false"` the sliced $B$ would be exactly zero instead — the filter would think the control does nothing at all.) Either way the sliced pair is a wrong model, not an approximate one.

Building the EKF model by re-formatting the template also beats parsing the XML and deleting `dyntype`/`dynprm` with `ElementTree`: `actrange` and `actearly` are only legal when an activation state exists, so an attribute-deletion pass has to know to strip those too or the compile fails.

**Process noise.** `model_ekf` does not contain the 4 ms motor lag that `model_sim` does. Inflate the velocity terms of $Q$ to cover that unmodelled lag, otherwise the filter will be overconfident during transient accelerations.

In [ ]:
def build_xml(*, motor_lag: bool) -> str:
    """Rellena el template con los parametros de arriba.

    motor_lag=True  -> actuadores con dyntype="filterexact": cada servo agrega un
                       estado de activacion a_i (lag de 1er orden, constante tau).
                       na = 2  ->  nx = 2*nv + na = 6.
    motor_lag=False -> mismos servos sin filtro: ctrl entra directo en la fuerza.
                       na = 0  ->  nx = 2*nv = 4.

    Nota: actrange y actearly SOLO son validos si hay estado de activacion, por eso
    viajan en el mismo campo que dyntype/dynprm y no como atributos sueltos.
    """
    fields = dict(geometry, **sim_options)

    for joint_name, joint in joints.items():
        lo_deg, hi_deg = joint["range_deg"]
        fields[f"{joint_name}_range_deg"] = f"{lo_deg} {hi_deg}"

    for idx, (act_name, act) in enumerate(actuators.items(), start=1):
        lo, hi = np.deg2rad(joints[act["joint"]]["range_deg"])
        # ctrl = angulo objetivo en rad -> el ctrlrange es el recorrido del joint.
        fields[f"{act_name}_ctrlrange"] = f"{lo:.6f} {hi:.6f}"
        fields[f"{act_name}_forcerange"] = f"{-act['force_max']} {act['force_max']}"
        fields[f"{act_name}_armature"] = act["armature"]
        fields[f"{act_name}_damping"] = act["damping"]
        # gainprm/biasprm derivados de kp y kv: un solo lugar por ganancia.
        #   force = gain_term*act + bias_term = kp*act + (-kp*angulo - kv*velocidad)
        fields[f"{act_name}_gainprm"] = f"{act['kp']} 0 0"
        fields[f"{act_name}_biasprm"] = f"0 {-act['kp']} {-act['kv']}"
        fields[f"motor_dynamics_{idx}"] = (
            f'dyntype="filterexact" dynprm="{act["tau"]}" actearly="true" '
            f'actrange="{lo:.6f} {hi:.6f}"'
            if motor_lag
            else ""
        )

    return xml_template.format(**fields)


def state_dim(m: mj.MjModel) -> int:
    """nx de MuJoCo: posiciones en el espacio tangente (nv, no nq) + activaciones."""
    return 2 * m.nv + m.na


# --- 1. PLANTA (alta fidelidad, 6 estados): mj_step + sensores sinteticos --------
model_sim = mj.MjModel.from_xml_string(build_xml(motor_lag=True))
data_sim = mj.MjData(model_sim)

# --- 2. MODELO DEL EKF (cuerpo rigido, 4 estados): mjd_transitionFD / mjd_inverseFD
model_ekf = mj.MjModel.from_xml_string(build_xml(motor_lag=False))
data_ekf = mj.MjData(model_ekf)

# OJO: MjModel no expone .nx en mujoco 3.x -> se calcula como 2*nv + na.
assert state_dim(model_sim) == 6, "La planta deberia tener 6 estados (q, v, a)"
assert state_dim(model_ekf) == 4, "El modelo del EKF deberia tener 4 estados (q, v)"
assert model_sim.nsensordata == model_ekf.nsensordata, "Los sensores deben coincidir"

print(f"model_sim -> nv={model_sim.nv}, na={model_sim.na}, nx={state_dim(model_sim)}")
print(f"model_ekf -> nv={model_ekf.nv}, na={model_ekf.na}, nx={state_dim(model_ekf)}")
print(f"bodies: {model_sim.nbody}, joints: {model_sim.njnt}, "
      f"geoms: {model_sim.ngeom}, sensores: {model_sim.nsensor}")

## Sine-driven trajectory: video + accelerometer readings

Driven on **`model_sim`** — the 6-state plant. Synthetic sensor data always comes from the plant, never from `model_ekf`.

Drive `joint_1` and `joint_2` along independent raised-cosine references — both start at angle 0 *and* angular velocity 0, matching the reset state, so there's no startup transient in the accelerometers.

The actuators are **position servos with realistic motor dynamics** (finite bandwidth via `dyntype="filterexact"`, stall-torque saturation via `forcerange`, reflected rotor inertia via `armature`, gearbox friction via `damping`) — all parameterised from the `actuators` dict. So `data_sim.ctrl` is the **target angle in radians**; the PD loop lives in the actuator definition and is integrated implicitly by MuJoCo, not hand-rolled in Python.

`data_sim.act` is logged alongside: it is the setpoint *after* the motor's first-order lag, i.e. the state `model_ekf` deliberately does not have.

In [ ]:
# Referencia por joint: coseno invertido ("raised cosine"), para que el angulo Y la
# velocidad angular arranquen en 0 -> coherente con el estado inicial que ya pone
# mj_resetData (qpos=0, qvel=0), sin pico de torque/aceleracion al arrancar.
#   theta(t)     = amplitude/2 * (1 - cos(w t))   -> theta(0) = 0
#   theta_dot(t) = amplitude/2 * w * sin(w t)      -> theta_dot(0) = 0
sine_params = {
    "joint_1": dict(amplitude=np.deg2rad(45), freq=0.5),
    "joint_2": dict(amplitude=np.deg2rad(60), freq=0.3),
}

def sine_target(p, t):
    w = 2 * np.pi * p["freq"]
    theta = p["amplitude"] / 2 * (1 - np.cos(w * t))
    theta_dot = p["amplitude"] / 2 * w * np.sin(w * t)
    return theta, theta_dot

# Ya NO hay PD en Python: los actuadores son servos de posicion (gaintype fixed +
# biastype affine), asi que data_sim.ctrl es el ANGULO OBJETIVO en rad. kp/kv,
# forcerange, armature y tau salen del dict `actuators`, se inyectan en el template
# y MuJoCo los integra implicitamente (integrator="implicitfast") -> mucho mas
# estable que el lazo hecho a mano.
# Se simula SIEMPRE con model_sim (la planta con lag de motor); model_ekf es solo
# para los jacobianos, nunca para generar datos.
print("Actuadores:", [model_sim.actuator(i).name for i in range(model_sim.nu)])
print("ctrlrange:\n", model_sim.actuator_ctrlrange)
print("forcerange:\n", model_sim.actuator_forcerange)

sim_duration = 6.0   # segundos
video_fps = 30
render_every = max(1, round(1 / (video_fps * model_sim.opt.timestep)))

In [ ]:
renderer = mj.Renderer(model_sim, height=480, width=640)
mj.mj_resetData(model_sim, data_sim)  # qpos = 0, qvel = 0 -> coincide con target(0), target_dot(0)

frames = []
log_time, log_qpos, log_qtarget = [], [], []
log_acc1, log_acc2, log_gyro1, log_gyro2 = [], [], [], []
log_torque, log_act = [], []

n_steps = int(sim_duration / model_sim.opt.timestep)
for step in range(n_steps):
    t = data_sim.time
    theta1_t, _ = sine_target(sine_params["joint_1"], t)
    theta2_t, _ = sine_target(sine_params["joint_2"], t)
    target = np.array([theta1_t, theta2_t])

    # ctrl = angulo objetivo (rad). El PD lo hace el servo, no nosotros.
    data_sim.ctrl[:] = target

    mj.mj_step(model_sim, data_sim)

    log_time.append(data_sim.time)
    log_qpos.append([data_sim.joint("joint_1").qpos[0], data_sim.joint("joint_2").qpos[0]])
    log_qtarget.append(target)
    log_act.append(data_sim.act.copy())               # setpoint filtrado (el estado extra)
    log_torque.append(data_sim.actuator_force.copy()) # torque real entregado (post-forcerange)
    log_acc1.append(data_sim.sensor("acc_i1").data.copy())
    log_acc2.append(data_sim.sensor("acc_i2").data.copy())
    log_gyro1.append(data_sim.sensor("gyro1").data.copy())
    log_gyro2.append(data_sim.sensor("gyro2").data.copy())

    if step % render_every == 0:
        renderer.update_scene(data_sim, camera="vista_plana")
        frames.append(renderer.render().copy())

renderer.close()

log_time = np.array(log_time)
log_qpos = np.array(log_qpos)
log_qtarget = np.array(log_qtarget)
log_act = np.array(log_act)
log_torque = np.array(log_torque)
log_acc1 = np.array(log_acc1)
log_acc2 = np.array(log_acc2)
log_gyro1 = np.array(log_gyro1)
log_gyro2 = np.array(log_gyro2)

err_deg = np.rad2deg(np.abs(log_qtarget - log_qpos)).max(axis=0)
lag_deg = np.rad2deg(np.abs(log_qtarget - log_act)).max(axis=0)
print(f"Simulados {len(log_time)} pasos ({log_time[-1]:.2f}s), {len(frames)} frames de video")
print(f"Error de seguimiento max: joint_1 {err_deg[0]:.2f} deg, joint_2 {err_deg[1]:.2f} deg")
# ctrl - act = lo que aporta SOLO el lag del motor. Es la parte de la dinamica que
# model_ekf no tiene, y la que hay que cubrir inflando Q en las velocidades.
print(f"Lag del motor (ctrl - act) max: joint_1 {lag_deg[0]:.2f} deg, joint_2 {lag_deg[1]:.2f} deg")
print(f"Torque pico: joint_1 {np.abs(log_torque[:, 0]).max():.4f} N*m "
      f"(limite {model_sim.actuator_forcerange[0, 1]:.2f}), "
      f"joint_2 {np.abs(log_torque[:, 1]).max():.4f} N*m "
      f"(limite {model_sim.actuator_forcerange[1, 1]:.2f})")

In [7]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.axis("off")
im = ax.imshow(frames[0])

def update(i):
    im.set_data(frames[i])
    return [im]

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / video_fps, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 16), sharex=True)

axes[0].plot(log_time, np.rad2deg(log_qtarget[:, 0]), "--", color="tab:blue", label="joint_1 target")
axes[0].plot(log_time, np.rad2deg(log_qpos[:, 0]), color="tab:blue", label="joint_1 actual")
axes[0].plot(log_time, np.rad2deg(log_qtarget[:, 1]), "--", color="tab:red", label="joint_2 target")
axes[0].plot(log_time, np.rad2deg(log_qpos[:, 1]), color="tab:red", label="joint_2 actual")
# act = setpoint despues del filtro del motor. La separacion target -> act es
# exactamente la dinamica que model_ekf NO modela.
axes[0].plot(log_time, np.rad2deg(log_act[:, 0]), ":", color="tab:blue", lw=1, label="joint_1 act (post-lag)")
axes[0].plot(log_time, np.rad2deg(log_act[:, 1]), ":", color="tab:red", lw=1, label="joint_2 act (post-lag)")
axes[0].set_ylabel("Angulo (deg)")
axes[0].set_title("Seguimiento de la referencia (servo de posicion)")
axes[0].legend(fontsize=8)
axes[0].grid(True)

# Torque entregado vs. torque de stall: muestra si el motor satura
axes[1].plot(log_time, log_torque[:, 0], color="tab:blue", label="act_joint_1")
axes[1].plot(log_time, log_torque[:, 1], color="tab:red", label="act_joint_2")
for i, color in enumerate(["tab:blue", "tab:red"]):
    axes[1].axhline(model_sim.actuator_forcerange[i, 1], color=color, ls=":", lw=1)
    axes[1].axhline(model_sim.actuator_forcerange[i, 0], color=color, ls=":", lw=1)
axes[1].set_ylabel("Torque (N*m)")
axes[1].set_title("Torque del actuador (punteado = forcerange / stall)")
axes[1].legend(fontsize=8)
axes[1].grid(True)

for k, axis in enumerate("xyz"):
    axes[2].plot(log_time, log_acc1[:, k], label=f"acc_i1_{axis}")
axes[2].set_ylabel("Aceleracion (m/s^2)")
axes[2].set_title("Acelerometro i1 (falange proximal)")
axes[2].legend()
axes[2].grid(True)

for k, axis in enumerate("xyz"):
    axes[3].plot(log_time, log_acc2[:, k], label=f"acc_i2_{axis}")
axes[3].set_ylabel("Aceleracion (m/s^2)")
axes[3].set_title("Acelerometro i2 (falange distal)")
axes[3].legend()
axes[3].grid(True)

for k, axis in enumerate("xyz"):
    axes[4].plot(log_time, log_gyro1[:, k], label=f"gyro1_{axis}")
for k, axis in enumerate("xyz"):
    axes[4].plot(log_time, log_gyro2[:, k], "--", label=f"gyro2_{axis}")
axes[4].set_xlabel("Tiempo (s)")
axes[4].set_ylabel("Velocidad angular (rad/s)")
axes[4].set_title("Giroscopios")
axes[4].legend(ncol=2, fontsize=8)
axes[4].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Linealizacion por diferencias finitas -> A, B, C, D del EKF.
#
# Se usa model_ekf (4 estados), NO model_sim (6 estados): ver la celda markdown de
# arriba. En el modelo de 6 estados u entra solo por las activaciones, asi que
# recortar el resultado a 4x4 da un modelo mal, no un modelo aproximado.
#
# mjd_transitionFD calcula CUATRO jacobianos, no dos, y la firma los exige a todos
# (pueden ser None, pero no se pueden omitir) -> pasar solo (A, B) da TypeError:
#     mjd_transitionFD(m, d, eps, flg_centered, A, B, C, D)
#         A = d(x_next)/dx   (nx, nx)      B = d(x_next)/du   (nx, nu)
#         C = d(sensor)/dx   (ns, nx)      D = d(sensor)/du   (ns, nu)

# 1. Punto de operacion EXPLICITO. Si no, se linealiza en el estado en que quedo el
#    loop de simulacion (t = 6 s), que no es un punto elegido a proposito.
#    En un EKF real esto se rehace en cada paso alrededor del estado estimado.
x_op = np.array([np.deg2rad(20), np.deg2rad(-15), 0.0, 0.0])  # [q1, q2, v1, v2]

mj.mj_resetData(model_ekf, data_ekf)
data_ekf.qpos[:] = x_op[:model_ekf.nv]
data_ekf.qvel[:] = x_op[model_ekf.nv:]
data_ekf.ctrl[:] = data_ekf.qpos     # setpoint del servo = pose actual (equilibrio)
mj.mj_forward(model_ekf, data_ekf)   # model_ekf.na == 0 -> no hay data.act que setear

# 2. Dimensiones. El estado de MuJoCo usa el espacio tangente (nv) para las
#    posiciones, asi maneja bien las cuaterniones: nx = 2*nv + na. Aca na = 0.
nx = state_dim(model_ekf)
nu = model_ekf.nu
ns = model_ekf.nsensordata
print(f"nv={model_ekf.nv}, na={model_ekf.na}  ->  nx={nx}, nu={nu}, nsensordata={ns}")

# 3. Salidas preasignadas: float64, C-contiguas y escribibles (np.zeros ya cumple).
A = np.zeros((nx, nx))
B = np.zeros((nx, nu))
C = np.zeros((ns, nx))
D = np.zeros((ns, nu))

# 4. Parametros de la diferenciacion finita
eps = 1e-6           # tamano de la perturbacion
flg_centered = True  # diferencias centradas: mas preciso, ~2x mas caro que forward

# 5. Calcular. La funcion maneja sola los warmstarts del solver y el clamping de ctrl.
mj.mjd_transitionFD(model_ekf, data_ekf, eps, flg_centered, A, B, C, D)

print(f"A (dx_next/dx)  {A.shape}")
print(f"B (dx_next/du)  {B.shape}")
print(f"C (dsensor/dx)  {C.shape}")
print(f"D (dsensor/du)  {D.shape}")

# C es justamente el jacobiano del modelo de medicion (la H del EKF): filas = canales
# de sensordata, columnas = estado. Ej.: las 3 filas del acelerometro de la falange
# proximal respecto del estado completo.
acc1_adr = model_ekf.sensor("acc_i1").adr[0]
print("\nH del acc_i1 (3 x nx) =\n", C[acc1_adr:acc1_adr + 3])

In [ ]:
np.set_printoptions(precision=6, suppress=True, linewidth=120)

print("A del EKF (4x4) =\n", A)
print("\nB del EKF (4x2) =\n", B)

# ---------------------------------------------------------------------------
# Por que NO se puede recortar el modelo de 6 estados: linealizamos la planta en
# el mismo punto de operacion y comparamos.
# ---------------------------------------------------------------------------
mj.mj_resetData(model_sim, data_sim)
data_sim.qpos[:] = x_op[:model_sim.nv]
data_sim.qvel[:] = x_op[model_sim.nv:]
data_sim.ctrl[:] = data_sim.qpos
data_sim.act[:] = data_sim.ctrl   # filtro del motor ya asentado -> equilibrio
mj.mj_forward(model_sim, data_sim)

nx_sim = state_dim(model_sim)
A_sim = np.zeros((nx_sim, nx_sim))
B_sim = np.zeros((nx_sim, model_sim.nu))
C_sim = np.zeros((model_sim.nsensordata, nx_sim))
D_sim = np.zeros((model_sim.nsensordata, model_sim.nu))
mj.mjd_transitionFD(model_sim, data_sim, eps, flg_centered, A_sim, B_sim, C_sim, D_sim)

print("\nB de la planta (6x2), recortada a sus 4 primeras filas =\n", B_sim[:4])

# El recorte no es cero (actearly="true" deja pasar parte del ctrl en el mismo paso),
# pero esta escalado por la ganancia de un paso del filtro: 1 - exp(-dt/tau).
one_step_gain = 1 - np.exp(-model_sim.opt.timestep / actuators["act_joint_1"]["tau"])
print(f"\nB_sim[:4] / B_ekf  ~  {np.median(B_sim[:4] / B):.6f}")
print(f"1 - exp(-dt/tau)   =  {one_step_gain:.6f}   <- coinciden: el recorte es B mal escalada")
print("Con actearly=false el recorte daria exactamente 0 (el EKF creeria que u no hace nada).")
print("Y ademas el recorte de A tira las columnas de 'a', que llevaban el resto del efecto.")